# Omnilex Colab Startup

This notebook sets up the Colab environment with persistent caching via Google Drive.

## ⚠️ One-Time Setup Required

Before using this notebook, you must populate the Drive cache once:
1. **Create cache directory** in your Google Drive: `MyDrive/omnilex-cache/`
2. **Populate uv wheel cache**: Run `uv pip install --all-extras` once, then copy `~/.cache/uv/` to `omnilex-cache/uv-cache/`
3. **Download Kaggle data**: Place competition data in `omnilex-cache/data/`
4. **Build indices**: Run the pipeline once, then copy `data/processed/` to `omnilex-cache/indices/`

See the **One-Time Setup Instructions** section at the end of this notebook for detailed steps.

In [ ]:
# @title Repository Configuration

repo_url = "https://github.com/your-username/Omnilex-Agentic-Retrieval-Competition.git"  # @param {type:"string"}  # noqa: E501
branch_name = "main"  # @param {type:"string"}

repo_dir = "/content/Omnilex-Agentic-Retrieval-Competition"

print(f"Repository: {repo_url}")
print(f"Branch: {branch_name}")
print(f"Local directory: {repo_dir}")

In [ ]:
# Step 1: Mount Google Drive
import os

from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/drive")

# Verify mount
if os.path.exists("/content/drive/MyDrive"):
    print("✓ Google Drive mounted successfully")
else:
    print("✗ Drive mount failed - please check authentication")
    raise RuntimeError("Drive mount failed")

In [ ]:
# Step 2: Configure uv cache (use local dir, pre-populated from Drive)
import os
import shutil
import subprocess
from pathlib import Path

drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
local_cache_dir = "/tmp/uv-cache"
archive_name = "uv-cache.tar.gz"
hash_name = "uv-cache.hash"

# Use local cache for uv (Drive doesn't support file locking)
os.environ["UV_CACHE_DIR"] = local_cache_dir
os.makedirs(local_cache_dir, exist_ok=True)

print(f"✓ UV_CACHE_DIR set to local path: {local_cache_dir}")

# Sync cached wheels from Drive to local cache using archive
drive_archive = os.path.join(drive_cache_dir, archive_name)
local_archive = os.path.join("/tmp", archive_name)

if os.path.exists(drive_archive):
    print(f"Found archive on Drive: {archive_name}")
    print("Copying archive from Drive (fast single-file read)...")

    # Copy archive from Drive to /tmp
    result = subprocess.run(["cp", drive_archive, local_archive], capture_output=True, text=True)

    if result.returncode == 0:
        print("  ✓ Archive copied to /tmp")

        # Verify archive integrity
        verify = subprocess.run(["gunzip", "-t", local_archive], capture_output=True, text=True)

        if verify.returncode == 0:
            print("  ✓ Archive verification passed")

            # Extract archive to local cache
            print(f"Extracting archive to {local_cache_dir}...")
            extract = subprocess.run(
                ["tar", "-xzf", local_archive, "-C", "/tmp/"], capture_output=True, text=True
            )

            if extract.returncode == 0:
                print("  ✓ Cache extracted from archive")
                # Clean up archive
                os.remove(local_archive)
            else:
                print(f"  ✗ Archive extraction failed: {extract.stderr}")
                print("  Falling back to PyPI downloads")
        else:
            print("  ⚠ Archive verification failed - may be corrupted")
            print("  Falling back to PyPI downloads")
            os.remove(local_archive) if os.path.exists(local_archive) else None
    else:
        print(f"  ✗ Failed to copy archive: {result.stderr}")
        print("  Falling back to PyPI downloads")
else:
    print(f"⚠ No archive found on Drive: {drive_archive}")
    print("  Will download packages from PyPI")
    print("  Tip: After first install, run the manual archive update cell")

# Display local cache size
if Path(local_cache_dir).exists():
    cache_size = (
        sum(f.stat().st_size for f in Path(local_cache_dir).rglob("*") if f.is_file()) / 1e6
    )
    print(f"Local cache size: {cache_size:.1f} MB")
else:
    print("Local cache empty")

In [ ]:
# Helper functions for uv cache archive management
import hashlib
import os
import subprocess
from pathlib import Path


def compute_cache_hash(cache_dir):
    """Compute MD5 hash of all files in cache directory."""
    md5 = hashlib.md5()
    if not os.path.exists(cache_dir):
        return None

    # Get all files, sort for consistency
    files = sorted(Path(cache_dir).rglob("*"), key=lambda p: str(p))
    for f in files:
        if f.is_file():
            # Hash relative path and file content
            rel_path = str(f.relative_to(cache_dir)).encode()
            md5.update(rel_path)
            try:
                with open(f, "rb") as fp:
                    # Read in chunks to handle large files
                    for chunk in iter(lambda: fp.read(8192), b""):
                        md5.update(chunk)
            except OSError:
                pass  # Skip files we can't read
    return md5.hexdigest()


def create_uv_cache_archive(cache_dir, output_path, hash_path=None):
    """Create tar.gz archive of uv cache and optional hash file."""
    try:
        # Check available space (need ~2x archive size for safety)
        stat = os.statvfs("/tmp")
        free_space = stat.f_bavail * stat.f_frsize / 1e9
        if free_space < 5:
            print(f"  ⚠ Warning: Low disk space ({free_space:.1f}GB free)")

        # Create archive
        print(f"Creating archive from {cache_dir}...")
        result = subprocess.run(
            ["tar", "-czf", str(output_path), "-C", "/tmp", "uv-cache"],
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            print(f"  ✗ Archive creation failed: {result.stderr}")
            return False

        # Verify archive
        verify = subprocess.run(["gunzip", "-t", str(output_path)], capture_output=True, text=True)

        if verify.returncode != 0:
            print("  ✗ Archive verification failed")
            return False

        archive_size = os.path.getsize(output_path) / 1e9
        print(f"  ✓ Archive created: {archive_size:.2f}GB")

        # Create hash file if requested
        if hash_path:
            cache_hash = compute_cache_hash(cache_dir)
            if cache_hash:
                with open(hash_path, "w") as f:
                    f.write(cache_hash)
                print(f"  ✓ Hash saved: {cache_hash[:16]}...")

        return True
    except Exception as e:
        print(f"  ✗ Error creating archive: {e}")
        return False


def copy_archive_to_drive(local_archive, drive_archive, local_hash=None, drive_hash=None):
    """Copy archive and optional hash to Drive."""
    try:
        # Copy archive
        result = subprocess.run(
            ["cp", local_archive, drive_archive], capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"  ✗ Failed to copy archive to Drive: {result.stderr}")
            return False

        print("  ✓ Archive copied to Drive")

        # Copy hash if provided
        if local_hash and drive_hash:
            result = subprocess.run(["cp", local_hash, drive_hash], capture_output=True, text=True)
            if result.returncode == 0:
                print("  ✓ Hash copied to Drive")

        return True
    except Exception as e:
        print(f"  ✗ Error copying to Drive: {e}")
        return False


print("✓ Archive helper functions loaded")

In [ ]:
# Step 3: Clone or update repository
import os
import subprocess

if not os.path.exists(repo_dir):
    print(f"Cloning repository to {repo_dir}...")
    result = subprocess.run(
        ["git", "clone", "-b", branch_name, repo_url, repo_dir], capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✓ Repository cloned successfully (branch: {branch_name})")
    else:
        print(f"✗ Clone failed: {result.stderr}")
        raise RuntimeError("Repository clone failed")
else:
    print(f"Repository exists, pulling updates (branch: {branch_name})...")
    os.chdir(repo_dir)
    result = subprocess.run(["git", "pull", "origin", branch_name], capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ Repository updated successfully")
    else:
        print(f"✗ Pull failed: {result.stderr}")
        print("  Continuing with existing code...")

# Change to repo directory
os.chdir(repo_dir)
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
# Step 4: Symlink data/ from Drive cache
import os

repo_dir = "/content/Omnilex-Agentic-Retrieval-Competition"
drive_data_cache = "/content/drive/MyDrive/omnilex-cache/data"
local_data_dir = os.path.join(repo_dir, "data")

# Check if Drive cache exists and has content
if os.path.exists(drive_data_cache) and os.listdir(drive_data_cache):
    # Remove existing data directory if it exists
    if os.path.exists(local_data_dir):
        if os.path.islink(local_data_dir):
            os.unlink(local_data_dir)
        else:
            import shutil

            shutil.rmtree(local_data_dir)

    # Create symlink
    os.symlink(drive_data_cache, local_data_dir)
    print(f"✓ Symlinked {local_data_dir} -> {drive_data_cache}")
else:
    print(f"⚠ Drive data cache empty or missing: {drive_data_cache}")
    print("  Skipping data symlink - will need to download data manually")

In [ ]:
# Step 5: Symlink data/processed/ (indices) from Drive cache
import os

repo_dir = "/content/Omnilex-Agentic-Retrieval-Competition"
drive_indices_cache = "/content/drive/MyDrive/omnilex-cache/indices"
local_processed_dir = os.path.join(repo_dir, "data", "processed")

# Check if Drive cache exists and has content
if os.path.exists(drive_indices_cache) and os.listdir(drive_indices_cache):
    # Ensure parent directory exists
    os.makedirs(os.path.join(repo_dir, "data"), exist_ok=True)

    # Remove existing processed directory if it exists
    if os.path.exists(local_processed_dir):
        if os.path.islink(local_processed_dir):
            os.unlink(local_processed_dir)
        else:
            import shutil

            shutil.rmtree(local_processed_dir)

    # Create symlink
    os.symlink(drive_indices_cache, local_processed_dir)
    print(f"✓ Symlinked {local_processed_dir} -> {drive_indices_cache}")
else:
    print(f"⚠ Drive indices cache empty or missing: {drive_indices_cache}")
    print("  Skipping indices symlink - will need to rebuild indices")

In [ ]:
# Step 6: Install dependencies using uv with local cache + auto-update archive
import os
import subprocess
from pathlib import Path

repo_dir = "/content/Omnilex-Agentic-Retrieval-Competition"
os.chdir(repo_dir)

drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
local_cache_dir = "/tmp/uv-cache"
archive_name = "uv-cache.tar.gz"
hash_name = "uv-cache.hash"

print("Installing dependencies with uv (using local cache)...")
print(f"UV_CACHE_DIR: {os.environ.get('UV_CACHE_DIR')}")

result = subprocess.run(["uv", "pip", "install", "-e", ".[dev]"], capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Dependencies installed successfully")
else:
    print(f"✗ Installation failed: {result.stderr}")
    # Try without -e flag as fallback
    print("\nTrying without editable install...")
    result = subprocess.run(["uv", "pip", "install", "."], capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ Dependencies installed successfully (non-editable)")
    else:
        print(f"✗ Installation failed: {result.stderr}")

# Auto-update archive if cache changed
print("\nChecking if uv cache changed...")
new_hash = compute_cache_hash(local_cache_dir)

if new_hash:
    drive_hash_path = os.path.join(drive_cache_dir, hash_name)

    # Read old hash from Drive
    old_hash = None
    if os.path.exists(drive_hash_path):
        with open(drive_hash_path) as f:
            old_hash = f.read().strip()

    if old_hash != new_hash:
        print("Cache changed detected!")
        print(f"  Old hash: {old_hash[:16] if old_hash else 'None'}...")
        print(f"  New hash: {new_hash[:16]}...")

        # Create new archive
        local_archive = os.path.join("/tmp", archive_name)
        local_hash = os.path.join("/tmp", hash_name)

        if create_uv_cache_archive(local_cache_dir, local_archive, local_hash):
            # Copy to Drive
            drive_archive = os.path.join(drive_cache_dir, archive_name)
            drive_hash_file = os.path.join(drive_cache_dir, hash_name)

            if copy_archive_to_drive(local_archive, drive_archive, local_hash, drive_hash_file):
                print("✓ Archive updated on Drive")

            # Cleanup local files
            os.remove(local_archive) if os.path.exists(local_archive) else None
            os.remove(local_hash) if os.path.exists(local_hash) else None
    else:
        print("✓ Cache unchanged, skipping archive update")
else:
    print("⚠ Could not compute cache hash")

In [ ]:
# Manual Archive Update Cell (run this to force-update the archive on Drive)
import os
import subprocess

drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
local_cache_dir = "/tmp/uv-cache"
archive_name = "uv-cache.tar.gz"
hash_name = "uv-cache.hash"

print("=== Manual Archive Update ===")

# Check if local cache exists
if not os.path.exists(local_cache_dir):
    print("✗ Local cache not found. Run Step 2 first.")
else:
    # Create archive
    local_archive = os.path.join("/tmp", archive_name)
    local_hash = os.path.join("/tmp", hash_name)

    if create_uv_cache_archive(local_cache_dir, local_archive, local_hash):
        # Verify archive
        verify = subprocess.run(["tar", "-tzf", local_archive], capture_output=True, text=True)

        if verify.returncode == 0:
            file_count = len(verify.stdout.strip().split("\n"))
            print(f"  ✓ Archive verified ({file_count} files)")

            # Copy to Drive
            drive_archive = os.path.join(drive_cache_dir, archive_name)
            drive_hash_file = os.path.join(drive_cache_dir, hash_name)

            if copy_archive_to_drive(local_archive, drive_archive, local_hash, drive_hash_file):
                print("✓ Archive manually updated on Drive")
        else:
            print("  ✗ Archive verification failed")

        # Cleanup
        os.remove(local_archive) if os.path.exists(local_archive) else None
        os.remove(local_hash) if os.path.exists(local_hash) else None
    else:
        print("✗ Failed to create archive")

In [ ]:
# Verification: Check setup
import os
import subprocess
from pathlib import Path

repo_dir = "/content/Omnilex-Agentic-Retrieval-Competition"

print("=== Setup Verification ===")

# Check Drive mount
drive_mounted = os.path.exists("/content/drive/MyDrive")
print(f"Drive mounted: {'✓' if drive_mounted else '✗'}")

# Check uv cache (local)
uv_cache = os.environ.get("UV_CACHE_DIR", "not set")
print(f"UV_CACHE_DIR (local): {uv_cache}")
if os.path.exists(uv_cache):
    cache_size = sum(f.stat().st_size for f in Path(uv_cache).rglob("*") if f.is_file()) / 1e6
    print(f"  Cache size: {cache_size:.1f} MB")

# Check Drive archive (new)
drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
archive_path = os.path.join(drive_cache_dir, "uv-cache.tar.gz")
hash_path = os.path.join(drive_cache_dir, "uv-cache.hash")

print(f"\nDrive archive exists: {'✓' if os.path.exists(archive_path) else '✗'}")
if os.path.exists(archive_path):
    archive_size = os.path.getsize(archive_path) / 1e9
    print(f"  Archive size: {archive_size:.2f} GB")

    # Verify archive integrity
    verify = subprocess.run(["gunzip", "-t", archive_path], capture_output=True, text=True)
    print(f"  Archive valid: {'✓' if verify.returncode == 0 else '✗'}")

print(f"Drive hash exists: {'✓' if os.path.exists(hash_path) else '✗'}")
if os.path.exists(hash_path):
    with open(hash_path) as f:
        print(f"  Hash: {f.read().strip()[:16]}...")

# Check repo
repo_exists = os.path.exists(repo_dir)
print(f"\nRepository exists: {'✓' if repo_exists else '✗'}")

# Check data symlink
data_dir = os.path.join(repo_dir, "data")
data_linked = os.path.islink(data_dir)
print(f"Data symlinked: {'✓' if data_linked else '✗'}")
if data_linked:
    print(f"  -> {os.readlink(data_dir)}")

# Check indices symlink
processed_dir = os.path.join(repo_dir, "data", "processed")
indices_linked = os.path.islink(processed_dir)
print(f"Indices symlinked: {'✓' if indices_linked else '✗'}")
if indices_linked:
    print(f"  -> {os.readlink(processed_dir)}")

print("\n=== Setup Complete ===")

## One-Time Setup Instructions

Follow these steps **once** to populate your Google Drive cache. After this, each new Colab runtime will use the cached data.

### Step 1: Create Cache Directory

In your Google Drive (MyDrive), create the following directory:
```
MyDrive/
  omnilex-cache/
```

### Step 2: Create uv Cache Archive

**New:** The notebook now uses a **tar.gz archive** for faster sync (~2-3 min vs ~10 min).

To create the initial archive:
1. Run this notebook once (it will download and install packages from PyPI)
2. After installation completes, **run the "Manual Archive Update" cell** in the notebook
   - This creates `uv-cache.tar.gz` (~2-3GB compressed) and `uv-cache.hash` on Drive
3. Future runs will use the archive for fast sync

**What gets created on Drive:**
- `MyDrive/omnilex-cache/uv-cache.tar.gz` - Compressed cache archive
- `MyDrive/omnilex-cache/uv-cache.hash` - Checksum for change detection

**How it works:**
- Notebook copies single archive file from Drive (fast FUSE read)
- Extracts archive to local `/tmp/uv-cache` (fast local disk I/O)
- After `uv pip install`, checksum detects changes and auto-updates archive
- `gunzip -t` verifies archive integrity before extraction

### Step 3: Download Kaggle Data

1. Go to [Kaggle Competition Page](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval/data)
2. Download the dataset
3. Upload the data files to `MyDrive/omnilex-cache/data/`

Required files typically include:
- `train.csv`
- `test.csv`
- Legal corpus files (SR, BGE documents)

### Step 4: Build and Cache Indices

1. Run the data processing pipeline once to build indices
2. After indices are built, copy them to Drive:
   ```python
   import os, shutil
   
   repo_dir = '/content/Omnilex-Agentic-Retrieval-Competition'
   indices_src = os.path.join(repo_dir, 'data', 'processed')
   indices_dst = '/content/drive/MyDrive/omnilex-cache/indices'
   
   if os.path.exists(indices_dst):
       shutil.rmtree(indices_dst)
   shutil.copytree(indices_src, indices_dst)
   print("✓ Indices copied to Drive")
   ```

### Step 5: Verify Cache

After populating the cache, verify in Drive:
- `omnilex-cache/uv-cache.tar.gz` should exist (~2-3GB compressed)
- `omnilex-cache/uv-cache.hash` should exist (checksum file)
- `omnilex-cache/data/` should contain Kaggle datasets (~4GB)
- `omnilex-cache/indices/` should contain pre-built indices (~1.5GB)

Total: ~8-9GB (fits in 10GB Drive budget with ~1GB headroom)

### Notes

- **GGUF Models**: Not cached (user will re-download from HuggingFace each session)
- **Archive Sync**: ~2-3 min (vs ~10 min for old `cp -r` approach)
- **Auto-Update**: Archive automatically updates when new packages are installed
- **Manual Update**: Run the "Manual Archive Update" cell to force-update the archive
- **Fallback**: If archive is missing/corrupted, notebook falls back to PyPI downloads